In [1]:
# @title
!mkdir -p ~/.kaggle
!cp kaggle/kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

# Test
!kaggle competitions list
!ls -l ~/.kaggle
!cat ~/.kaggle/kaggle.json
!bash setup.sh

ref                                                                                 deadline             category                reward  teamCount  userHasEntered  
----------------------------------------------------------------------------------  -------------------  ---------------  -------------  ---------  --------------  
https://www.kaggle.com/competitions/ai-mathematical-olympiad-progress-prize-3       2026-04-15 23:59:00  Featured         2,207,152 Usd        797           False  
https://www.kaggle.com/competitions/vesuvius-challenge-surface-detection            2026-02-13 23:59:00  Research           200,000 Usd        349           False  
https://www.kaggle.com/competitions/google-tunix-hackathon                          2026-01-12 23:59:00  Featured           100,000 Usd         84           False  
https://www.kaggle.com/competitions/csiro-biomass                                   2026-01-28 23:59:00  Research            75,000 Usd       2096           False  
https://ww

In [2]:
"""
PART 5: THE OMNI-ENSEMBLE (Merging Titan + Grand)
--------------------------------------------------
Goal: Break 0.415 by combining ALL trained architectures.

Strategy:
1. Define ALL model architectures used so far (B3, Dense121, ResNet50, ResNeXt50, Dense169).
2. Scan and LOAD every .pth checkpoint found on disk.
   - Grand Team: B3, DenseNet121, ResNet50 (~15 models)
   - Titan Team: ResNeXt50, DenseNet169 (~8 models)
3. INFERENCE: Average ~23 models using 8-View TTA.

NOW ALSO TRAINABLE:
- If any checkpoint is missing, it will train it using the SAME training recipes you used before,
  save the .pth, then proceed to Omni-Ensemble inference + eval + final stats.
"""

import os
import re
import time
import numpy as np
import pandas as pd
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image, ImageOps
from tqdm import tqdm
import warnings

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

warnings.filterwarnings('ignore')


# ---------------- CONFIGURATION ---------------- #
class Config:
    TRAIN_DATA_PATH = "data/train_data"
    TEST_DATA_PATH = "data/test_data"
    TRAIN_LABELS_PATH = "data/train_labels.csv"

    # Settings (inference / eval)
    BATCH_SIZE = 16
    IMG_SIZE = 512
    NUM_CLASSES = 4

    # CUDA/MPS/CPU compatible
    DEVICE = torch.device(
        "cuda" if torch.cuda.is_available()
        else ("mps" if torch.backends.mps.is_available() else "cpu")
    )

    NUM_WORKERS = 4

    CLASS_NAMES = ['HER2(+)', 'Luminal A', 'Luminal B', 'Triple negative']
    SEEDS = [42, 2024, 7, 33]


# ---------------- UTILITIES ---------------- #

def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)

def find_mask_filename(image_name, all_files):
    base_id = re.findall(r'\d+', image_name)[0]
    candidate1 = f"mask_{base_id}.png"
    if candidate1 in all_files: return candidate1
    candidate2 = f"img_{base_id}_mask.png"
    if candidate2 in all_files: return candidate2
    return None

def crop_to_mask(image, mask):
    mask_arr = np.array(mask)
    rows = np.any(mask_arr, axis=1)
    cols = np.any(mask_arr, axis=0)
    if not np.any(rows) or not np.any(cols): return image
    ymin, ymax = np.where(rows)[0][[0, -1]]
    xmin, xmax = np.where(cols)[0][[0, -1]]
    pad = 20
    ymin = max(0, ymin - pad);
    ymax = min(mask_arr.shape[0], ymax + pad)
    xmin = max(0, xmin - pad);
    xmax = min(mask_arr.shape[1], xmax + pad)
    cropped_img = image.crop((xmin, ymin, xmax, ymax))
    w, h = cropped_img.size
    target_size = max(w, h)
    padding = ((target_size - w) // 2, (target_size - h) // 2, target_size - w - (target_size - w) // 2,
               target_size - h - (target_size - h) // 2)
    return ImageOps.expand(cropped_img, padding, fill=(255, 255, 255))


# ---------------- DATASET & TRANSFORMS ---------------- #

def get_transforms(is_train=True):
    # Only need validation transforms for inference (as in your omni code)
    return transforms.Compose([
        transforms.Resize((Config.IMG_SIZE, Config.IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

# TRAIN transforms (same style you used across the training parts)
def get_train_transforms():
    return transforms.Compose([
        transforms.Resize((Config.IMG_SIZE, Config.IMG_SIZE)),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomApply([transforms.RandomRotation((90, 90))], p=0.5),
        transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.02),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])


class SmartCroppingDataset(Dataset):
    def __init__(self, data_path, labels_df=None, transform=None, is_test=False):
        self.data_path = Path(data_path)
        self.transform = transform
        self.is_test = is_test
        self.all_files = set(os.listdir(data_path))
        self.image_files = sorted([f for f in self.all_files if f.startswith('img_') and not 'mask' in f])
        if not is_test and labels_df is not None:
            self.label_map = {name: idx for idx, name in enumerate(Config.CLASS_NAMES)}
            self.labels_dict = dict(zip(labels_df['sample_index'], labels_df['label']))
            self.image_files = [f for f in self.image_files if f in self.labels_dict]

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = self.data_path / img_name
        image = Image.open(img_path).convert('RGB')
        mask_name = find_mask_filename(img_name, self.all_files)
        if mask_name:
            mask = Image.open(self.data_path / mask_name).convert('L')
            image = crop_to_mask(image, mask)
        if self.transform: image = self.transform(image)
        if self.is_test:
            return image, img_name
        else:
            return image, self.label_map[self.labels_dict[img_name]]


# ---------------- MODEL ARCHITECTURES (ALL) ---------------- #

# 1. EfficientNet B3
class HistologyNet(nn.Module):
    def __init__(self, num_classes=4):
        super(HistologyNet, self).__init__()
        self.backbone = models.efficientnet_b3(weights=None)
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(0.5), nn.Linear(in_features, 512), nn.SiLU(),
            nn.Dropout(0.3), nn.Linear(512, num_classes)
        )
    def forward(self, x): return self.backbone(x)

# 2. DenseNet121 (Old & New)
class DenseNet121(nn.Module):
    def __init__(self, num_classes=4):
        super(DenseNet121, self).__init__()
        self.backbone = models.densenet121(weights=None)
        in_features = self.backbone.classifier.in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(in_features, num_classes)
        )
    def forward(self, x): return self.backbone(x)

# 3. ResNet50
class ResNet50(nn.Module):
    def __init__(self, num_classes=4):
        super(ResNet50, self).__init__()
        self.backbone = models.resnet50(weights=None)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_features, num_classes)
        )
    def forward(self, x): return self.backbone(x)

# 4. ResNeXt50 (From Titan)
class ResNeXt50(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.backbone = models.resnext50_32x4d(weights=None)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(in_features, num_classes))
    def forward(self, x): return self.backbone(x)

# 5. DenseNet169 (From Titan)
class DenseNet169(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.backbone = models.densenet169(weights=None)
        in_features = self.backbone.classifier.in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(in_features, num_classes)
        )
    def forward(self, x): return self.backbone(x)


# ---------------- TRAINING UTILS (SAME RECIPES) ---------------- #

# Focal Loss (used in your B3 + DenseV2 training scripts)
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma
    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, reduction='none')
        pt = torch.exp(-ce)
        loss = ((1 - pt) ** self.gamma) * ce
        return loss.mean()

# Mixup helpers (used in your mixup scripts)
def mixup_data(x, y, alpha=1.0, device=None):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(device if device is not None else x.device)
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

def _get_train_loader(batch_size, shuffle=True, pin_memory=True):
    train_labels = pd.read_csv(Config.TRAIN_LABELS_PATH)
    ds = SmartCroppingDataset(Config.TRAIN_DATA_PATH, train_labels, transform=get_train_transforms(), is_test=False)
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=Config.NUM_WORKERS,
        pin_memory=pin_memory,
        persistent_workers=True if Config.NUM_WORKERS > 0 else False
    )

# --- EXACT TRAIN RECIPES (matching your prior parts) ---

# B3: Focal + Mixup + AdamW + OneCycleLR (like your B3 mixup run)
def train_b3_seed(seed):
    set_seed(seed)
    print(f"\n🚀 Training MISSING: B3 Seed {seed} (Focal + Mixup + OneCycle) ...")
    BATCH_SIZE = 14
    NUM_EPOCHS = 25
    LEARNING_RATE = 2e-4
    MIXUP_ALPHA = 0.4

    loader = _get_train_loader(BATCH_SIZE, shuffle=True, pin_memory=True)
    model = HistologyNet(Config.NUM_CLASSES).to(Config.DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-3)
    criterion = FocalLoss(gamma=2.0)
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=LEARNING_RATE,
        steps_per_epoch=len(loader),
        epochs=NUM_EPOCHS,
        pct_start=0.3
    )

    best_loss = float('inf')
    for epoch in range(NUM_EPOCHS):
        model.train()
        running_loss = 0.0
        pbar = tqdm(loader, desc=f"B3 Ep {epoch+1}", leave=False)
        for images, labels in pbar:
            images, labels = images.to(Config.DEVICE), labels.to(Config.DEVICE)
            optimizer.zero_grad()

            if np.random.random() < 0.6:
                images, ta, tb, lam = mixup_data(images, labels, MIXUP_ALPHA, device=Config.DEVICE)
                outputs = model(images)
                loss = mixup_criterion(criterion, outputs, ta, tb, lam)
            else:
                outputs = model(images)
                loss = criterion(outputs, labels)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

            running_loss += loss.item()
            pbar.set_postfix({'loss': f"{loss.item():.4f}"})

        epoch_loss = running_loss / len(loader)
        if epoch > 15 and epoch_loss < best_loss:
            best_loss = epoch_loss
            torch.save(model.state_dict(), f"model_seed_{seed}.pth")

    if not os.path.exists(f"model_seed_{seed}.pth"):
        torch.save(model.state_dict(), f"model_seed_{seed}.pth")

    print(f"✅ Saved: model_seed_{seed}.pth")
    return

# DenseNet121 OLD: CE + label_smoothing + AdamW + OneCycleLR
def train_densenet_old_seed(seed):
    set_seed(seed)
    print(f"\n🚀 Training MISSING: DenseNet121 OLD Seed {seed} (CE + smoothing) ...")
    BATCH_SIZE = 12
    NUM_EPOCHS = 25
    LEARNING_RATE = 1e-4

    loader = _get_train_loader(BATCH_SIZE, shuffle=True, pin_memory=True)
    model = DenseNet121(Config.NUM_CLASSES).to(Config.DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-3)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=LEARNING_RATE,
        steps_per_epoch=len(loader),
        epochs=NUM_EPOCHS
    )

    best_loss = float('inf')
    for epoch in range(NUM_EPOCHS):
        model.train()
        running_loss = 0.0
        for images, labels in loader:
            images, labels = images.to(Config.DEVICE), labels.to(Config.DEVICE)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            scheduler.step()
            running_loss += loss.item()

        epoch_loss = running_loss / len(loader)
        if epoch > 10 and epoch_loss < best_loss:
            best_loss = epoch_loss
            torch.save(model.state_dict(), f"densenet_seed_{seed}.pth")

    print(f"✅ Saved: densenet_seed_{seed}.pth")
    return

# DenseNet121 V2: Focal + Mixup + AdamW + OneCycleLR
def train_densenet_v2_seed(seed):
    set_seed(seed)
    print(f"\n🚀 Training MISSING: DenseNet121 V2 Seed {seed} (Focal + Mixup) ...")
    BATCH_SIZE = 10
    NUM_EPOCHS = 35
    LEARNING_RATE = 2e-4
    MIXUP_ALPHA = 1.0

    loader = _get_train_loader(BATCH_SIZE, shuffle=True, pin_memory=True)
    model = DenseNet121(Config.NUM_CLASSES).to(Config.DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-3)
    criterion = FocalLoss(gamma=2.0)
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=LEARNING_RATE,
        steps_per_epoch=len(loader),
        epochs=NUM_EPOCHS
    )

    best_loss = float('inf')
    for epoch in range(NUM_EPOCHS):
        model.train()
        running_loss = 0.0
        for images, labels in loader:
            images, labels = images.to(Config.DEVICE), labels.to(Config.DEVICE)
            optimizer.zero_grad()

            if np.random.random() < 0.6:
                images, ta, tb, lam = mixup_data(images, labels, MIXUP_ALPHA, device=Config.DEVICE)
                outputs = model(images)
                loss = mixup_criterion(criterion, outputs, ta, tb, lam)
            else:
                outputs = model(images)
                loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()
            scheduler.step()
            running_loss += loss.item()

        epoch_loss = running_loss / len(loader)
        if epoch > 15 and epoch_loss < best_loss:
            best_loss = epoch_loss
            torch.save(model.state_dict(), f"densenet_v2_seed_{seed}.pth")

    print(f"✅ Saved: densenet_v2_seed_{seed}.pth")
    return

# ResNet50: CE + smoothing + AdamW + OneCycleLR
def train_resnet_seed(seed):
    set_seed(seed)
    print(f"\n🚀 Training MISSING: ResNet50 Seed {seed} (CE + smoothing) ...")
    BATCH_SIZE = 12
    NUM_EPOCHS = 25
    LEARNING_RATE = 2e-4

    loader = _get_train_loader(BATCH_SIZE, shuffle=True, pin_memory=True)
    model = ResNet50(Config.NUM_CLASSES).to(Config.DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-3)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=LEARNING_RATE,
        steps_per_epoch=len(loader),
        epochs=NUM_EPOCHS
    )

    best_loss = float('inf')
    for epoch in range(NUM_EPOCHS):
        model.train()
        running_loss = 0.0
        for images, labels in loader:
            images, labels = images.to(Config.DEVICE), labels.to(Config.DEVICE)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            scheduler.step()
            running_loss += loss.item()

        epoch_loss = running_loss / len(loader)
        if epoch > 10 and epoch_loss < best_loss:
            best_loss = epoch_loss
            torch.save(model.state_dict(), f"resnet_seed_{seed}.pth")

    print(f"✅ Saved: resnet_seed_{seed}.pth")
    return

# Titan: Mixup 100% + CE smoothing + AdamW + OneCycleLR (ResNeXt & Dense169)
def train_titan_model(model_class, seed, name):
    set_seed(seed)
    print(f"\n🚀 Training MISSING: {name} Seed {seed} (Mixup 100%) ...")
    BATCH_SIZE = 12
    NUM_EPOCHS = 18
    LEARNING_RATE = 2e-4
    MIXUP_ALPHA = 0.4

    save_path = f"{name}_seed_{seed}.pth"

    loader = _get_train_loader(BATCH_SIZE, shuffle=True, pin_memory=True)
    model = model_class(Config.NUM_CLASSES).to(Config.DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-3)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=LEARNING_RATE,
        steps_per_epoch=len(loader),
        epochs=NUM_EPOCHS
    )

    best_loss = float('inf')
    for epoch in range(NUM_EPOCHS):
        model.train()
        running_loss = 0.0
        for images, labels in loader:
            images, labels = images.to(Config.DEVICE), labels.to(Config.DEVICE)
            optimizer.zero_grad()

            images, ta, tb, lam = mixup_data(images, labels, MIXUP_ALPHA, device=Config.DEVICE)
            outputs = model(images)
            loss = mixup_criterion(criterion, outputs, ta, tb, lam)

            loss.backward()
            optimizer.step()
            scheduler.step()
            running_loss += loss.item()

        epoch_loss = running_loss / len(loader)
        if epoch_loss < best_loss:
            best_loss = epoch_loss
            torch.save(model.state_dict(), save_path)

    print(f"✅ Saved: {save_path}")
    return


# ---------------- EVAL HELPERS ---------------- #

def _print_eval_stats(y_true, y_pred, title="EVAL"):
    acc = accuracy_score(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred)
    rec_macro = recall_score(y_true, y_pred, average="macro", zero_division=0)
    rec_weighted = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
    f1_weighted = f1_score(y_true, y_pred, average="weighted", zero_division=0)

    per_class_recall = recall_score(y_true, y_pred, average=None, zero_division=0)
    cm = confusion_matrix(y_true, y_pred)

    print("\n" + "=" * 72)
    print(f"[{title}] STATS")
    print("=" * 72)
    print(f"Accuracy:           {acc:.6f}")
    print(f"Balanced Accuracy:  {bacc:.6f}")
    print(f"Recall (MACRO):     {rec_macro:.6f}   <-- main")
    print(f"Recall (WEIGHTED):  {rec_weighted:.6f}")
    print(f"F1 (MACRO):         {f1_macro:.6f}")
    print(f"F1 (WEIGHTED):      {f1_weighted:.6f}")

    print("\nPer-class Recall:")
    for i, name in enumerate(Config.CLASS_NAMES):
        val = float(per_class_recall[i]) if i < len(per_class_recall) else 0.0
        print(f"  {name:15s}: {val:.6f}")

    print("\nClassification Report:")
    print(classification_report(
        y_true, y_pred,
        target_names=Config.CLASS_NAMES,
        digits=6,
        zero_division=0
    ))

    print("Confusion Matrix (rows=true, cols=pred):")
    print(cm)

def evaluate_omni_on_labeled_data(all_models, data_loader, augmentations, title="TRAIN (IN-SAMPLE)"):
    y_true = []
    y_pred = []

    with torch.no_grad():
        for images, labels in tqdm(data_loader, desc=f"Evaluating ({title})"):
            images = images.to(Config.DEVICE)
            labels = labels.cpu().numpy()
            y_true.extend(labels.tolist())

            batch_ensemble_probs = []
            for model in all_models:
                tta_preds = []
                for aug in augmentations:
                    p = torch.softmax(model(aug(images)), dim=1)
                    tta_preds.append(p)
                model_avg = torch.stack(tta_preds).mean(dim=0)
                batch_ensemble_probs.append(model_avg)

            grand_avg = torch.stack(batch_ensemble_probs).mean(dim=0)
            preds = torch.argmax(grand_avg, dim=1).cpu().numpy()
            y_pred.extend(preds.tolist())

    _print_eval_stats(np.array(y_true), np.array(y_pred), title=title)


# ---------------- MAIN ---------------- #

def main():
    print(f"Stats: OMNI-ENSEMBLE (Train missing -> Load ALL -> TTA -> Submit)")
    start_time = time.time()

    # Sanity for training
    if not (os.path.exists(Config.TRAIN_LABELS_PATH) and os.path.exists(Config.TRAIN_DATA_PATH)):
        print("CRITICAL: train_labels.csv or train_data missing -> cannot train missing checkpoints.")
    else:
        # TRAIN MISSING CHECKPOINTS (same filenames your omni loader expects)
        for seed in Config.SEEDS:
            if not os.path.exists(f"model_seed_{seed}.pth"):
                train_b3_seed(seed)

            if not os.path.exists(f"densenet_seed_{seed}.pth"):
                train_densenet_old_seed(seed)

            if not os.path.exists(f"densenet_v2_seed_{seed}.pth"):
                train_densenet_v2_seed(seed)

            if not os.path.exists(f"resnet_seed_{seed}.pth"):
                train_resnet_seed(seed)

            if not os.path.exists(f"resnext_mixup_seed_{seed}.pth") and not os.path.exists(f"resnext_seed_{seed}.pth"):
                train_titan_model(ResNeXt50, seed, "resnext_mixup")

            if not os.path.exists(f"densenet169_mixup_seed_{seed}.pth") and not os.path.exists(f"densenet169_seed_{seed}.pth"):
                train_titan_model(DenseNet169, seed, "densenet169_mixup")

    all_models = []

    # Helper to load models
    def load_if_exists(filename, model_class, name):
        if os.path.exists(filename):
            try:
                m = model_class(Config.NUM_CLASSES)
                m.load_state_dict(torch.load(filename, map_location=Config.DEVICE))
                m.to(Config.DEVICE).eval()
                all_models.append(m)
                return True
            except Exception as e:
                print(f"Failed to load {filename}: {e}")
        return False

    # --- TEAM GRAND ---
    print("\nLoading Team Grand (B3, Dense121, ResNet50)...")
    count_grand = 0
    for seed in Config.SEEDS:
        if load_if_exists(f"model_seed_{seed}.pth", HistologyNet, "B3"): count_grand += 1
        if load_if_exists(f"densenet_seed_{seed}.pth", DenseNet121, "Dense121_Old"): count_grand += 1
        if load_if_exists(f"densenet_v2_seed_{seed}.pth", DenseNet121, "Dense121_New"): count_grand += 1
        if load_if_exists(f"resnet_seed_{seed}.pth", ResNet50, "ResNet50"): count_grand += 1
    print(f"-> Loaded {count_grand} models.")

    # --- TEAM TITAN ---
    print("\nLoading Team Titan (ResNeXt, Dense169)...")
    count_titan = 0
    for seed in Config.SEEDS:
        if load_if_exists(f"resnext_mixup_seed_{seed}.pth", ResNeXt50, "ResNeXt_Mixup"):
            count_titan += 1
        elif load_if_exists(f"resnext_seed_{seed}.pth", ResNeXt50, "ResNeXt"):
            count_titan += 1

        if load_if_exists(f"densenet169_mixup_seed_{seed}.pth", DenseNet169, "Dense169_Mixup"):
            count_titan += 1
        elif load_if_exists(f"densenet169_seed_{seed}.pth", DenseNet169, "Dense169"):
            count_titan += 1
    print(f"-> Loaded {count_titan} models.")

    print(f"\nTotal Omni-Ensemble Size: {len(all_models)} models")
    if len(all_models) == 0:
        print("CRITICAL: No models found. Please run previous training parts first.")
        return

    # 3. Omni-Inference
    test_dataset = SmartCroppingDataset(Config.TEST_DATA_PATH, transform=get_transforms(False), is_test=True)
    test_loader = DataLoader(test_dataset, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=Config.NUM_WORKERS)

    all_probs = []
    sample_indices = []

    print("\nRunning Omni-Inference (8-View TTA)...")

    augmentations = [
        lambda x: x,
        lambda x: torch.rot90(x, 1, [2, 3]),
        lambda x: torch.rot90(x, 2, [2, 3]),
        lambda x: torch.rot90(x, 3, [2, 3]),
        lambda x: transforms.functional.hflip(x),
        lambda x: torch.rot90(transforms.functional.hflip(x), 1, [2, 3]),
        lambda x: torch.rot90(transforms.functional.hflip(x), 2, [2, 3]),
        lambda x: torch.rot90(transforms.functional.hflip(x), 3, [2, 3]),
    ]

    with torch.no_grad():
        for images, filenames in tqdm(test_loader):
            images = images.to(Config.DEVICE)
            sample_indices.extend(filenames)

            batch_ensemble_probs = []

            for model in all_models:
                tta_preds = []
                for aug in augmentations:
                    p = torch.softmax(model(aug(images)), dim=1)
                    tta_preds.append(p)

                model_avg = torch.stack(tta_preds).mean(dim=0)
                batch_ensemble_probs.append(model_avg)

            grand_avg = torch.stack(batch_ensemble_probs).mean(dim=0)
            all_probs.append(grand_avg.cpu().numpy())

    final_probs = np.concatenate(all_probs)
    predictions = np.argmax(final_probs, axis=1)

    submission = pd.DataFrame({
        'sample_index': sample_indices,
        'label': [Config.CLASS_NAMES[p] for p in predictions]
    })

    submission.to_csv('submission_omni_ensemble.csv', index=False)
    print("\nDone! Saved to 'submission_omni_ensemble.csv'")
    print("This file contains the combined wisdom of all your runs. 🧠")

    # ---------------- ADDED EVALUATION (IF LABELS EXIST) ---------------- #
    if os.path.exists(Config.TRAIN_LABELS_PATH) and os.path.exists(Config.TRAIN_DATA_PATH):
        try:
            labels_df = pd.read_csv(Config.TRAIN_LABELS_PATH)
            train_dataset = SmartCroppingDataset(
                Config.TRAIN_DATA_PATH,
                labels_df=labels_df,
                transform=get_transforms(False),
                is_test=False
            )
            train_loader = DataLoader(
                train_dataset,
                batch_size=Config.BATCH_SIZE,
                shuffle=False,
                num_workers=Config.NUM_WORKERS
            )

            print("\nRunning Omni-Evaluation on TRAIN (IN-SAMPLE) (8-View TTA)...")
            evaluate_omni_on_labeled_data(
                all_models=all_models,
                data_loader=train_loader,
                augmentations=augmentations,
                title="TRAIN (IN-SAMPLE)"
            )
        except Exception as e:
            print(f"\n[WARN] Could not run evaluation on labeled train set: {e}")

    # ---------------- FINAL STATS ---------------- #
    elapsed = time.time() - start_time
    print("\n" + "=" * 72)
    print("[FINAL STATS]")
    print("=" * 72)
    print(f"Device:                   {Config.DEVICE}")
    print(f"IMG_SIZE:                 {Config.IMG_SIZE}")
    print(f"Batch Size:               {Config.BATCH_SIZE}")
    print(f"Loaded Team Grand Models: {count_grand}")
    print(f"Loaded Team Titan Models: {count_titan}")
    print(f"Total Models:             {len(all_models)}")
    print(f"Test Samples:             {len(test_dataset)}")
    print(f"Time Elapsed (sec):       {elapsed:.2f}")
    print("=" * 72)


if __name__ == "__main__":
    main()

Stats: OMNI-ENSEMBLE (Train missing -> Load ALL -> TTA -> Submit)

🚀 Training MISSING: B3 Seed 42 (Focal + Mixup + OneCycle) ...


✅ Saved: model_seed_42.pth

🚀 Training MISSING: DenseNet121 OLD Seed 42 (CE + smoothing) ...
✅ Saved: densenet_seed_42.pth

🚀 Training MISSING: DenseNet121 V2 Seed 42 (Focal + Mixup) ...
✅ Saved: densenet_v2_seed_42.pth

🚀 Training MISSING: ResNet50 Seed 42 (CE + smoothing) ...
✅ Saved: resnet_seed_42.pth

🚀 Training MISSING: resnext_mixup Seed 42 (Mixup 100%) ...
✅ Saved: resnext_mixup_seed_42.pth

🚀 Training MISSING: densenet169_mixup Seed 42 (Mixup 100%) ...
✅ Saved: densenet169_mixup_seed_42.pth

🚀 Training MISSING: B3 Seed 2024 (Focal + Mixup + OneCycle) ...


✅ Saved: model_seed_2024.pth

🚀 Training MISSING: DenseNet121 OLD Seed 2024 (CE + smoothing) ...
✅ Saved: densenet_seed_2024.pth

🚀 Training MISSING: DenseNet121 V2 Seed 2024 (Focal + Mixup) ...
✅ Saved: densenet_v2_seed_2024.pth

🚀 Training MISSING: ResNet50 Seed 2024 (CE + smoothing) ...
✅ Saved: resnet_seed_2024.pth

🚀 Training MISSING: resnext_mixup Seed 2024 (Mixup 100%) ...
✅ Saved: resnext_mixup_seed_2024.pth

🚀 Training MISSING: densenet169_mixup Seed 2024 (Mixup 100%) ...
✅ Saved: densenet169_mixup_seed_2024.pth

🚀 Training MISSING: B3 Seed 7 (Focal + Mixup + OneCycle) ...


✅ Saved: model_seed_7.pth

🚀 Training MISSING: DenseNet121 OLD Seed 7 (CE + smoothing) ...
✅ Saved: densenet_seed_7.pth

🚀 Training MISSING: DenseNet121 V2 Seed 7 (Focal + Mixup) ...
✅ Saved: densenet_v2_seed_7.pth

🚀 Training MISSING: ResNet50 Seed 7 (CE + smoothing) ...
✅ Saved: resnet_seed_7.pth

🚀 Training MISSING: resnext_mixup Seed 7 (Mixup 100%) ...
✅ Saved: resnext_mixup_seed_7.pth

🚀 Training MISSING: densenet169_mixup Seed 7 (Mixup 100%) ...
✅ Saved: densenet169_mixup_seed_7.pth

🚀 Training MISSING: B3 Seed 33 (Focal + Mixup + OneCycle) ...


✅ Saved: model_seed_33.pth

🚀 Training MISSING: DenseNet121 OLD Seed 33 (CE + smoothing) ...
✅ Saved: densenet_seed_33.pth

🚀 Training MISSING: DenseNet121 V2 Seed 33 (Focal + Mixup) ...
✅ Saved: densenet_v2_seed_33.pth

🚀 Training MISSING: ResNet50 Seed 33 (CE + smoothing) ...
✅ Saved: resnet_seed_33.pth

🚀 Training MISSING: resnext_mixup Seed 33 (Mixup 100%) ...
✅ Saved: resnext_mixup_seed_33.pth

🚀 Training MISSING: densenet169_mixup Seed 33 (Mixup 100%) ...
✅ Saved: densenet169_mixup_seed_33.pth

Loading Team Grand (B3, Dense121, ResNet50)...
-> Loaded 16 models.

Loading Team Titan (ResNeXt, Dense169)...
-> Loaded 8 models.

Total Omni-Ensemble Size: 24 models

Running Omni-Inference (8-View TTA)...


100%|██████████| 30/30 [13:49<00:00, 27.66s/it]



Done! Saved to 'submission_omni_ensemble.csv'
This file contains the combined wisdom of all your runs. 🧠

Running Omni-Evaluation on TRAIN (IN-SAMPLE) (8-View TTA)...


Evaluating (TRAIN (IN-SAMPLE)): 100%|██████████| 44/44 [20:01<00:00, 27.30s/it]


[TRAIN (IN-SAMPLE)] STATS
Accuracy:           0.435601
Balanced Accuracy:  0.374929
Recall (MACRO):     0.374929   <-- main
Recall (WEIGHTED):  0.435601
F1 (MACRO):         0.359560
F1 (WEIGHTED):      0.389816

Per-class Recall:
  HER2(+)        : 0.211640
  Luminal A      : 0.253659
  Luminal B      : 0.904545
  Triple negative: 0.129870

Classification Report:
                 precision    recall  f1-score   support

        HER2(+)   0.851064  0.211640  0.338983       189
      Luminal A   0.641975  0.253659  0.363636       205
      Luminal B   0.363139  0.904545  0.518229       220
Triple negative   0.666667  0.129870  0.217391        77

       accuracy                       0.435601       691
      macro avg   0.630711  0.374929  0.359560       691
   weighted avg   0.613140  0.435601  0.389816       691

Confusion Matrix (rows=true, cols=pred):
[[ 40  10 138   1]
 [  3  52 149   1]
 [  4  14 199   3]
 [  0   5  62  10]]

[FINAL STATS]
Device:                   cuda
IMG_SIZE: 

In [6]:
# standalone_eval_model_based.py
# ------------------------------------------------------------
# Prints MODEL-BASED stats:
# 1) Each single checkpoint alone (with 8-view TTA)
# 2) Per-team aggregates (Grand only, Titan only)
# 3) Full Omni ensemble (all checkpoints)
# Also saves:
# - model_stats.csv
# - per_class_stats_per_model/*.csv  (one file per model)
# ------------------------------------------------------------

import os
import re
import numpy as np
import pandas as pd
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image, ImageOps
from tqdm import tqdm

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    recall_score,
    f1_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
)

# ---------------- CONFIG ----------------
class Config:
    TRAIN_DATA_PATH = "data/train_data"
    TRAIN_LABELS_PATH = "data/train_labels.csv"

    BATCH_SIZE = 16
    IMG_SIZE = 512
    NUM_CLASSES = 4
    NUM_WORKERS = 4

    DEVICE = torch.device(
        "cuda" if torch.cuda.is_available()
        else ("mps" if torch.backends.mps.is_available() else "cpu")
    )

    CLASS_NAMES = ['HER2(+)', 'Luminal A', 'Luminal B', 'Triple negative']
    SEEDS = [42, 2024, 7, 33]

    SAVE_PER_CLASS_CSV_PER_MODEL = True
    PER_CLASS_DIR = "per_class_stats_per_model"


# ---------------- SAME CROPPING PIPELINE ----------------
def find_mask_filename(image_name, all_files):
    base_id = re.findall(r'\d+', image_name)[0]
    candidate1 = f"mask_{base_id}.png"
    if candidate1 in all_files: return candidate1
    candidate2 = f"img_{base_id}_mask.png"
    if candidate2 in all_files: return candidate2
    return None

def crop_to_mask(image, mask):
    mask_arr = np.array(mask)
    rows = np.any(mask_arr, axis=1)
    cols = np.any(mask_arr, axis=0)
    if not np.any(rows) or not np.any(cols):
        return image
    ymin, ymax = np.where(rows)[0][[0, -1]]
    xmin, xmax = np.where(cols)[0][[0, -1]]
    pad = 20
    ymin = max(0, ymin - pad)
    ymax = min(mask_arr.shape[0], ymax + pad)
    xmin = max(0, xmin - pad)
    xmax = min(mask_arr.shape[1], xmax + pad)
    cropped_img = image.crop((xmin, ymin, xmax, ymax))

    w, h = cropped_img.size
    target_size = max(w, h)
    padding = (
        (target_size - w) // 2,
        (target_size - h) // 2,
        target_size - w - (target_size - w) // 2,
        target_size - h - (target_size - h) // 2
    )
    return ImageOps.expand(cropped_img, padding, fill=(255, 255, 255))

def get_transforms():
    return transforms.Compose([
        transforms.Resize((Config.IMG_SIZE, Config.IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

class SmartCroppingDataset(Dataset):
    def __init__(self, data_path, labels_df=None, transform=None):
        self.data_path = Path(data_path)
        self.transform = transform
        self.all_files = set(os.listdir(data_path))
        self.image_files = sorted([f for f in self.all_files if f.startswith('img_') and 'mask' not in f])

        self.label_map = {name: idx for idx, name in enumerate(Config.CLASS_NAMES)}
        self.labels_dict = dict(zip(labels_df['sample_index'], labels_df['label']))
        self.image_files = [f for f in self.image_files if f in self.labels_dict]

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = self.data_path / img_name
        image = Image.open(img_path).convert('RGB')

        mask_name = find_mask_filename(img_name, self.all_files)
        if mask_name:
            mask = Image.open(self.data_path / mask_name).convert('L')
            image = crop_to_mask(image, mask)

        if self.transform:
            image = self.transform(image)

        label = self.label_map[self.labels_dict[img_name]]
        return image, label


# ---------------- MODEL DEFINITIONS (same as your omni) ----------------
class HistologyNet(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.backbone = models.efficientnet_b3(weights=None)
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(0.5), nn.Linear(in_features, 512), nn.SiLU(),
            nn.Dropout(0.3), nn.Linear(512, num_classes)
        )
    def forward(self, x): return self.backbone(x)

class DenseNet121(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.backbone = models.densenet121(weights=None)
        in_features = self.backbone.classifier.in_features
        self.backbone.classifier = nn.Sequential(nn.Dropout(0.5), nn.Linear(in_features, num_classes))
    def forward(self, x): return self.backbone(x)

class ResNet50(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.backbone = models.resnet50(weights=None)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(in_features, num_classes))
    def forward(self, x): return self.backbone(x)

class ResNeXt50(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.backbone = models.resnext50_32x4d(weights=None)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(in_features, num_classes))
    def forward(self, x): return self.backbone(x)

class DenseNet169(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.backbone = models.densenet169(weights=None)
        in_features = self.backbone.classifier.in_features
        self.backbone.classifier = nn.Sequential(nn.Dropout(0.5), nn.Linear(in_features, num_classes))
    def forward(self, x): return self.backbone(x)


# ---------------- TTA ----------------
augmentations = [
    lambda x: x,
    lambda x: torch.rot90(x, 1, [2, 3]),
    lambda x: torch.rot90(x, 2, [2, 3]),
    lambda x: torch.rot90(x, 3, [2, 3]),
    lambda x: transforms.functional.hflip(x),
    lambda x: torch.rot90(transforms.functional.hflip(x), 1, [2, 3]),
    lambda x: torch.rot90(transforms.functional.hflip(x), 2, [2, 3]),
    lambda x: torch.rot90(transforms.functional.hflip(x), 3, [2, 3]),
]

def load_model(path, model_cls):
    m = model_cls(Config.NUM_CLASSES)
    m.load_state_dict(torch.load(path, map_location=Config.DEVICE))
    m.to(Config.DEVICE).eval()
    return m

@torch.no_grad()
def predict_single_model(model, loader):
    y_true, y_pred = [], []
    for images, labels in loader:
        images = images.to(Config.DEVICE)
        y_true.extend(labels.numpy().tolist())

        tta_probs = []
        for aug in augmentations:
            logits = model(aug(images))
            probs = torch.softmax(logits, dim=1)
            tta_probs.append(probs)
        avg_probs = torch.stack(tta_probs).mean(dim=0)
        preds = torch.argmax(avg_probs, dim=1).cpu().numpy().tolist()
        y_pred.extend(preds)

    return np.array(y_true), np.array(y_pred)

@torch.no_grad()
def predict_ensemble(models_list, loader):
    y_true, y_pred = [], []
    for images, labels in loader:
        images = images.to(Config.DEVICE)
        y_true.extend(labels.numpy().tolist())

        per_model_probs = []
        for m in models_list:
            tta_probs = []
            for aug in augmentations:
                logits = m(aug(images))
                probs = torch.softmax(logits, dim=1)
                tta_probs.append(probs)
            per_model_probs.append(torch.stack(tta_probs).mean(dim=0))

        avg_probs = torch.stack(per_model_probs).mean(dim=0)
        preds = torch.argmax(avg_probs, dim=1).cpu().numpy().tolist()
        y_pred.extend(preds)

    return np.array(y_true), np.array(y_pred)


# ---------------- METRICS ----------------
def compute_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred)
    rec_macro = recall_score(y_true, y_pred, average="macro", zero_division=0)   # main
    rec_weighted = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
    f1_weighted = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    return acc, bacc, rec_macro, rec_weighted, f1_macro, f1_weighted

def per_class_df(y_true, y_pred):
    prec, rec, f1, sup = precision_recall_fscore_support(
        y_true, y_pred, labels=list(range(len(Config.CLASS_NAMES))), zero_division=0
    )
    pred_counts = np.bincount(y_pred, minlength=len(Config.CLASS_NAMES))
    df = pd.DataFrame({
        "class": Config.CLASS_NAMES,
        "support": sup.astype(int),
        "predicted_count": pred_counts.astype(int),
        "precision": prec,
        "recall": rec,
        "f1": f1,
    })
    return df


# ---------------- BUILD MODEL LIST (each checkpoint separately) ----------------
def enumerate_checkpoints():
    entries = []

    # Grand
    for seed in Config.SEEDS:
        p = f"model_seed_{seed}.pth"
        if os.path.exists(p): entries.append(("Grand", f"B3_seed_{seed}", p, HistologyNet))

        p = f"densenet_seed_{seed}.pth"
        if os.path.exists(p): entries.append(("Grand", f"Dense121_old_seed_{seed}", p, DenseNet121))

        p = f"densenet_v2_seed_{seed}.pth"
        if os.path.exists(p): entries.append(("Grand", f"Dense121_v2_seed_{seed}", p, DenseNet121))

        p = f"resnet_seed_{seed}.pth"
        if os.path.exists(p): entries.append(("Grand", f"ResNet50_seed_{seed}", p, ResNet50))

    # Titan
    for seed in Config.SEEDS:
        p1 = f"resnext_mixup_seed_{seed}.pth"
        p2 = f"resnext_seed_{seed}.pth"
        if os.path.exists(p1): entries.append(("Titan", f"ResNeXt50_mixup_seed_{seed}", p1, ResNeXt50))
        elif os.path.exists(p2): entries.append(("Titan", f"ResNeXt50_seed_{seed}", p2, ResNeXt50))

        p1 = f"densenet169_mixup_seed_{seed}.pth"
        p2 = f"densenet169_seed_{seed}.pth"
        if os.path.exists(p1): entries.append(("Titan", f"Dense169_mixup_seed_{seed}", p1, DenseNet169))
        elif os.path.exists(p2): entries.append(("Titan", f"Dense169_seed_{seed}", p2, DenseNet169))

    return entries


def main():
    print(f"Device: {Config.DEVICE}")

    # data
    labels_df = pd.read_csv(Config.TRAIN_LABELS_PATH)
    ds = SmartCroppingDataset(Config.TRAIN_DATA_PATH, labels_df=labels_df, transform=get_transforms())
    loader = DataLoader(ds, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=Config.NUM_WORKERS)

    ckpts = enumerate_checkpoints()
    print(f"Found checkpoints: {len(ckpts)}")
    if len(ckpts) == 0:
        print("CRITICAL: No checkpoints found in current directory.")
        return

    if Config.SAVE_PER_CLASS_CSV_PER_MODEL:
        os.makedirs(Config.PER_CLASS_DIR, exist_ok=True)

    rows = []

    # ---- MODEL-BY-MODEL ----
    for team, name, path, cls in tqdm(ckpts, desc="Evaluating each model"):
        model = load_model(path, cls)
        y_true, y_pred = predict_single_model(model, loader)
        acc, bacc, rec_macro, rec_weighted, f1_macro, f1_weighted = compute_metrics(y_true, y_pred)

        rows.append({
            "team": team,
            "model": name,
            "path": path,
            "accuracy": acc,
            "balanced_accuracy": bacc,
            "recall_macro": rec_macro,
            "recall_weighted": rec_weighted,
            "f1_macro": f1_macro,
            "f1_weighted": f1_weighted,
        })

        if Config.SAVE_PER_CLASS_CSV_PER_MODEL:
            dfc = per_class_df(y_true, y_pred)
            safe = re.sub(r"[^a-zA-Z0-9_\-]+", "_", name)
            dfc.to_csv(os.path.join(Config.PER_CLASS_DIR, f"{safe}.csv"), index=False)

    df_models = pd.DataFrame(rows).sort_values("recall_macro", ascending=False)
    df_models.to_csv("model_stats.csv", index=False)

    print("\n" + "=" * 90)
    print("MODEL-BASED RANKING (sorted by Recall MACRO = main)")
    print("=" * 90)
    with pd.option_context("display.max_rows", 200, "display.max_columns", 200, "display.width", 200):
        print(df_models.to_string(index=False, justify="left",
                                  formatters={
                                      "accuracy": "{:.6f}".format,
                                      "balanced_accuracy": "{:.6f}".format,
                                      "recall_macro": "{:.6f}".format,
                                      "recall_weighted": "{:.6f}".format,
                                      "f1_macro": "{:.6f}".format,
                                      "f1_weighted": "{:.6f}".format,
                                  }))

    print("\nSaved: model_stats.csv")
    if Config.SAVE_PER_CLASS_CSV_PER_MODEL:
        print(f"Saved per-class CSVs: {Config.PER_CLASS_DIR}/<model>.csv")

    # ---- TEAM AGGREGATES ----
    # Grand-only ensemble, Titan-only ensemble, Full omni ensemble
    def load_team_models(team_name):
        ms = []
        for team, name, path, cls in ckpts:
            if team == team_name:
                ms.append(load_model(path, cls))
        return ms

    print("\n" + "=" * 90)
    print("TEAM / ENSEMBLE STATS")
    print("=" * 90)

    for label, models_list in [
        ("Grand_Ensemble", load_team_models("Grand")),
        ("Titan_Ensemble", load_team_models("Titan")),
        ("Omni_Ensemble", [load_model(path, cls) for _, _, path, cls in ckpts]),
    ]:
        if len(models_list) == 0:
            print(f"{label}: no models")
            continue

        y_true, y_pred = predict_ensemble(models_list, loader)
        acc, bacc, rec_macro, rec_weighted, f1_macro, f1_weighted = compute_metrics(y_true, y_pred)

        print(f"\n[{label}] models={len(models_list)}")
        print(f"Accuracy:          {acc:.6f}")
        print(f"Balanced Accuracy: {bacc:.6f}")
        print(f"Recall (MACRO):    {rec_macro:.6f}   <-- main")
        print(f"Recall (WEIGHTED): {rec_weighted:.6f}")
        print(f"F1 (MACRO):        {f1_macro:.6f}")
        print(f"F1 (WEIGHTED):     {f1_weighted:.6f}")

        # per-class table for this ensemble
        dfc = per_class_df(y_true, y_pred)
        with pd.option_context("display.max_rows", 200, "display.max_columns", 200, "display.width", 200):
            print("\nPer-class:")
            print(dfc.to_string(index=False, justify="left",
                               formatters={
                                   "precision": "{:.6f}".format,
                                   "recall": "{:.6f}".format,
                                   "f1": "{:.6f}".format
                               }))

    # Optional: print full report for best single model
    best = df_models.iloc[0]
    print("\n" + "=" * 90)
    print("BEST SINGLE MODEL (full report)")
    print("=" * 90)
    best_team, best_name, best_path = best["team"], best["model"], best["path"]
    best_entry = [e for e in ckpts if e[1] == best_name and e[2] == best_path][0]
    _, _, path, cls = best_entry
    best_model = load_model(path, cls)
    y_true, y_pred = predict_single_model(best_model, loader)
    print(classification_report(y_true, y_pred, target_names=Config.CLASS_NAMES, digits=6, zero_division=0))
    print("Confusion Matrix (rows=true, cols=pred):")
    print(confusion_matrix(y_true, y_pred))


if __name__ == "__main__":
    main()

Device: cuda
Found checkpoints: 24


Evaluating each model: 100%|██████████| 24/24 [20:59<00:00, 52.47s/it]



MODEL-BASED RANKING (sorted by Recall MACRO = main)
team  model                     path                            accuracy balanced_accuracy recall_macro recall_weighted f1_macro f1_weighted
Grand      Dense121_old_seed_33            densenet_seed_33.pth 0.463097 0.396743          0.396743     0.463097        0.393226 0.438986   
Grand    Dense121_old_seed_2024          densenet_seed_2024.pth 0.452967 0.392961          0.392961     0.452967        0.387263 0.421512   
Grand       Dense121_old_seed_7             densenet_seed_7.pth 0.448625 0.387186          0.387186     0.448625        0.374420 0.410051   
Grand     Dense121_v2_seed_2024       densenet_v2_seed_2024.pth 0.431259 0.384374          0.384374     0.431259        0.385147 0.410936   
Grand      Dense121_old_seed_42            densenet_seed_42.pth 0.442836 0.378612          0.378612     0.442836        0.368194 0.407587   
Grand        Dense121_v2_seed_7          densenet_v2_seed_7.pth 0.435601 0.376853          0.376853  